# 🧠 OmniFile AI Processor — Colab Diagnostic & Testing Notebook
**المطور:** Dr Abdulmalek Tamer Al-husseini | **الإصدار:** v4.0 | **البيئة:** Google Colab + Gradio

> هذا الـ notebook يختبر المشروع وحدةً وحدة، ثم يشغّل واجهة Gradio قابلة للوصول من الجوال.
> 
> **ترتيب التشغيل:** نفّذ الخلايا بالترتيب من الأعلى إلى الأسفل.


## 📥 الخطوة 1 — استنساخ المشروع وإعداد البيئة

In [ ]:
# ✅ استنساخ المشروع
import os, sys

REPO_URL = 'https://github.com/DrAbdulmalek/OmniFile_Processor.git'
REPO_DIR = '/content/OmniFile_Processor'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print('✅ تم الاستنساخ')
else:
    !cd {REPO_DIR} && git pull
    print('🔄 تم التحديث')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'📂 المجلد الحالي: {os.getcwd()}')
!echo '📁 محتوى المشروع:' && ls -la

## 📦 الخطوة 2 — تثبيت التبعيات (مرحلية)

In [ ]:
# المرحلة الأولى: أساسيات خفيفة — تعمل دائماً
!pip install -q gradio>=4.0 Pillow numpy difflib2 python-docx openpyxl 2>/dev/null || true
!pip install -q arabic-reshaper python-bidi langdetect 2>/dev/null || true
print('✅ الحزم الأساسية جاهزة')

In [ ]:
# المرحلة الثانية: OCR (قد تأخذ دقائق)
!pip install -q easyocr pytesseract 2>/dev/null || true
!apt-get install -qq tesseract-ocr tesseract-ocr-ara 2>/dev/null || true
print('✅ محركات OCR جاهزة')

In [ ]:
# المرحلة الثالثة: NLP ثقيلة (اختياري — تحتاج وقتاً)
INSTALL_HEAVY = False  # ← غيّر إلى True إذا أردت transformers كاملة
if INSTALL_HEAVY:
    !pip install -q transformers torch sentencepiece sacremoses 2>/dev/null || true
    print('✅ NLP الثقيلة جاهزة')
else:
    print('⏭️ تم تخطي الحزم الثقيلة (INSTALL_HEAVY=False)')

## 🩺 الخطوة 3 — فحص صحة المشروع

In [ ]:
import os, importlib, traceback

# قائمة الوحدات المتوقعة
expected_modules = [
    ('modules.core.structure',       'النماذج الأساسية'),
    ('modules.vision.ocr_engine',    'محرك OCR'),
    ('modules.vision.image_preprocessor', 'معالجة الصور'),
    ('modules.vision.layout_analyzer','تحليل التخطيط'),
    ('modules.nlp.arabic_rtl',       'RTL العربية'),
    ('modules.nlp.spell_corrector',  'تدقيق إملائي'),
    ('modules.nlp.language_detector','كاشف اللغة'),
    ('modules.nlp.arabic_nlp_utils', 'NLP عربي (جديد)'),
    ('modules.export.exporter',      'التصدير'),
    ('modules.export.layout_preserving','تصدير محافظ على التخطيط (جديد)'),
    ('modules.evaluation.metrics',   'مقاييس الجودة'),
    ('modules.security.encryption',  'التشفير'),
]

print('='*55)
print('🩺 فحص وحدات المشروع')
print('='*55)
ok, fail = [], []
for mod, label in expected_modules:
    try:
        importlib.import_module(mod)
        print(f'  ✅ {label:<30} [{mod}]')
        ok.append(mod)
    except Exception as e:
        print(f'  ❌ {label:<30} [{mod}]')
        print(f'     └─ {type(e).__name__}: {str(e)[:80]}')
        fail.append((mod, str(e)))

print('='*55)
print(f'النتيجة: {len(ok)} نجح | {len(fail)} فشل | المجموع {len(expected_modules)}')
print('='*55)
HEALTH_OK = len(fail) < len(expected_modules) // 2
print('الحالة العامة:', '🟢 مقبول' if HEALTH_OK else '🔴 مشكلة جوهرية')

## 🧪 الخطوة 4 — اختبارات الوحدات

In [ ]:
# ── اختبار 4.1: معالجة النص العربي ──────────────────────────────────
print('='*50)
print('🔤 اختبار 4.1 — معالجة النص العربي')
print('='*50)

test_texts = [
    ('بسيط', 'مرحبا بالعالم'),
    ('تشكيل', 'الطَّبيبُ يَعمَلُ في المُستشفى'),
    ('همزات', 'الأطباء والأسرة والإنسان'),
    ('مختلط', 'OCR النتيجة: ٩٥٪ دقة'),
    ('طبي',  'كسر في الفخذ Femur Fracture'),
]

try:
    from modules.nlp.arabic_rtl import ArabicRTLProcessor
    proc = ArabicRTLProcessor()
    for label, text in test_texts:
        result = proc.process(text)
        print(f'  [{label}] {text[:30]}')
        print(f'      → {str(result)[:60]}')
    print('✅ ArabicRTLProcessor يعمل')
except Exception as e:
    print(f'❌ {e}')

# اختبار arabic_nlp_utils (الجديد)
try:
    from modules.nlp.arabic_nlp_utils import normalize_for_comparison, similarity_report
    t1 = 'الأطبّاء يعملون'
    t2 = 'الاطباء يعملون'
    report = similarity_report(t1, t2)
    print(f'\n🆕 arabic_nlp_utils:')
    print(f'   "{t1}" vs "{t2}"')
    print(f'   التشابه الخام:      {report["raw_similarity"]:.3f}')
    print(f'   التشابه المُطبَّع:   {report["normalized_similarity"]:.3f}')
    print(f'   التوصية:            {report["recommendation"]}')
    print('✅ arabic_nlp_utils يعمل')
except Exception as e:
    print(f'❌ arabic_nlp_utils: {e}')

In [ ]:
# ── اختبار 4.2: OCR على صورة تجريبية ────────────────────────────────
print('='*50)
print('🖼️  اختبار 4.2 — OCR')
print('='*50)

import numpy as np
from PIL import Image, ImageDraw, ImageFont
import io

# إنشاء صورة اختبارية
def make_test_image(text='OmniFile Test مرحبا', size=(400,100)):
    img = Image.new('RGB', size, color='white')
    d = ImageDraw.Draw(img)
    d.text((10, 30), text, fill='black')
    return img

test_img = make_test_image()
test_img.save('/tmp/test_ocr.png')
print('📸 صورة اختبارية أُنشئت: /tmp/test_ocr.png')

# تجربة OCR
try:
    import easyocr
    reader = easyocr.Reader(['ar', 'en'], gpu=False, verbose=False)
    result = reader.readtext('/tmp/test_ocr.png')
    print(f'EasyOCR → {result}')
    print('✅ EasyOCR يعمل')
except Exception as e:
    print(f'⚠️ EasyOCR: {e}')

try:
    import pytesseract
    text = pytesseract.image_to_string(test_img, lang='ara+eng')
    print(f'Tesseract → {repr(text.strip())}')
    print('✅ Tesseract يعمل')
except Exception as e:
    print(f'⚠️ Tesseract: {e}')

In [ ]:
# ── اختبار 4.3: التصدير ──────────────────────────────────────────────
print('='*50)
print('📤 اختبار 4.3 — التصدير')
print('='*50)

try:
    from modules.export.layout_preserving import export_to_docx
    layout_data = {
        'blocks': [
            {'type': 'header',    'bbox': [0,0,1,.1],  'text': 'تقرير اختبار OmniFile'},
            {'type': 'paragraph', 'bbox': [0,.1,1,.4], 'text': 'هذا نص تجريبي لاختبار التصدير مع الحفاظ على التخطيط.'},
            {'type': 'caption',   'bbox': [0,.4,1,.5], 'text': 'ملاحظة: تم الإنشاء تلقائياً'},
            {'type': 'table',     'bbox': [0,.5,1,.9],
             'cells': [['الوحدة','الحالة','الدقة'],
                       ['OCR','✅ يعمل','95%'],
                       ['NLP','✅ يعمل','98%'],
                       ['Export','✅ يعمل','100%']]},
        ]
    }
    out = export_to_docx(layout_data, '/tmp/test_layout.docx')
    size = os.path.getsize(out)
    print(f'✅ layout_preserving → {out} ({size:,} bytes)')
except Exception as e:
    print(f'❌ layout_preserving: {e}')

try:
    from modules.evaluation.metrics import compute_cer, compute_wer
    cer = compute_cer('مرحبا بالعالم', 'مرحبا العالم')
    wer = compute_wer('hello world test', 'hello word test')
    print(f'✅ metrics → CER={cer:.3f} | WER={wer:.3f}')
except Exception as e:
    print(f'⚠️ metrics: {e}')

In [ ]:
# ── اختبار 4.4: الأمان والتشفير ─────────────────────────────────────
print('='*50)
print('🔒 اختبار 4.4 — التشفير')
print('='*50)

try:
    from modules.security.encryption import FileEncryptor
    enc = FileEncryptor()
    original = b'OmniFile test data - بيانات اختبار'
    with open('/tmp/test_plain.txt', 'wb') as f:
        f.write(original)
    enc.encrypt_file('/tmp/test_plain.txt', '/tmp/test_encrypted.enc')
    enc.decrypt_file('/tmp/test_encrypted.enc', '/tmp/test_decrypted.txt')
    recovered = open('/tmp/test_decrypted.txt', 'rb').read()
    assert recovered == original, 'البيانات لا تتطابق!'
    print('✅ التشفير/فك التشفير يعمل بشكل صحيح')
    print(f'   الأصل:     {original[:40]}')
    print(f'   المُسترجع: {recovered[:40]}')
except Exception as e:
    print(f'⚠️ التشفير: {e}')

## 🎛️ الخطوة 5 — واجهة Gradio للاختبار التفاعلي
> هذه الواجهة تعمل على الحاسوب والجوال معاً.  
> بعد تشغيلها ستحصل على رابط عام (public URL) صالح لساعتين.


In [ ]:
import gradio as gr
import numpy as np
from PIL import Image
import json, os, traceback, time

# ── مساعدات ────────────────────────────────────────────────────────────
def safe_import(mod, cls=None):
    try:
        m = __import__(mod, fromlist=[cls] if cls else [])
        return getattr(m, cls) if cls else m, None
    except Exception as e:
        return None, str(e)

# ── Tab 1: OCR ─────────────────────────────────────────────────────────
def run_ocr(image, engine_choice, preprocess):
    if image is None:
        return '⚠️ لم يتم رفع صورة', None
    logs = [f'⏱️ {time.strftime("%H:%M:%S")} — بدء OCR ({engine_choice})']
    result_text = ''
    try:
        pil_img = Image.fromarray(image) if isinstance(image, np.ndarray) else image
        if preprocess:
            import cv2
            arr = np.array(pil_img.convert('L'))
            _, arr = cv2.threshold(arr, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            pil_img = Image.fromarray(arr)
            logs.append('🔧 معالجة مسبقة: Otsu thresholding')
        tmp = '/tmp/_gradio_ocr_in.png'
        pil_img.save(tmp)
        if engine_choice == 'EasyOCR':
            import easyocr
            reader = easyocr.Reader(['ar','en'], gpu=False, verbose=False)
            res = reader.readtext(tmp)
            result_text = '\n'.join([r[1] for r in res])
            logs.append(f'📊 {len(res)} كتلة نصية')
        elif engine_choice == 'Tesseract':
            import pytesseract
            result_text = pytesseract.image_to_string(pil_img, lang='ara+eng')
        elif engine_choice == 'OmniFile Engine':
            OCREngine, err = safe_import('modules.vision.ocr_engine', 'OCREngine')
            if OCREngine:
                engine = OCREngine()
                result = engine.process(tmp)
                result_text = getattr(result, 'text', str(result))
            else:
                result_text = f'⚠️ OCREngine: {err}'
        logs.append(f'✅ النتيجة ({len(result_text)} حرف)')
    except Exception as e:
        logs.append(f'❌ {type(e).__name__}: {e}')
        result_text = str(e)
    return result_text, '\n'.join(logs)

# ── Tab 2: NLP ─────────────────────────────────────────────────────────
def run_nlp(text, operation):
    if not text.strip():
        return '⚠️ أدخل نصاً', ''
    logs = [f'⏱️ {time.strftime("%H:%M:%S")} — {operation}']
    result = ''
    try:
        if operation == 'تطبيع عربي':
            from modules.nlp.arabic_nlp_utils import normalize_for_comparison
            result = normalize_for_comparison(text)
        elif operation == 'تقرير تشابه':
            lines = text.strip().split('\n')
            if len(lines) >= 2:
                from modules.nlp.arabic_nlp_utils import similarity_report
                rep = similarity_report(lines[0], lines[1])
                result = json.dumps(rep, ensure_ascii=False, indent=2)
            else:
                result = '⚠️ أدخل سطرين (نص1 ثم نص2)'
        elif operation == 'RTL معالجة':
            ArabicRTL, err = safe_import('modules.nlp.arabic_rtl', 'ArabicRTLProcessor')
            if ArabicRTL:
                result = str(ArabicRTL().process(text))
            else:
                result = f'⚠️ {err}'
        elif operation == 'كشف لغة':
            from langdetect import detect
            lang = detect(text)
            result = f'اللغة المكتشفة: {lang}'
        elif operation == 'تدقيق إملائي':
            SpellCorrector, err = safe_import('modules.nlp.spell_corrector', 'SpellCorrector')
            if SpellCorrector:
                result = SpellCorrector().correct(text)
            else:
                result = f'⚠️ {err}'
        logs.append(f'✅ اكتمل ({len(result)} حرف)')
    except Exception as e:
        logs.append(f'❌ {traceback.format_exc()}')
        result = str(e)
    return result, '\n'.join(logs)

# ── Tab 3: Export ──────────────────────────────────────────────────────
def run_export(text, fmt):
    if not text.strip():
        return None, '⚠️ أدخل نصاً'
    logs = [f'⏱️ {time.strftime("%H:%M:%S")} — تصدير {fmt}']
    out_path = f'/tmp/omnifile_export.{fmt.lower()}'
    try:
        if fmt == 'DOCX':
            from modules.export.layout_preserving import export_to_docx
            layout = {'blocks': [{'type':'paragraph','bbox':[0,0,1,1],'text':text}]}
            export_to_docx(layout, out_path)
        elif fmt == 'TXT':
            with open(out_path, 'w', encoding='utf-8-sig') as f:
                f.write(text)
        elif fmt == 'JSON':
            out_path = '/tmp/omnifile_export.json'
            with open(out_path, 'w', encoding='utf-8') as f:
                json.dump({'text': text, 'length': len(text)}, f, ensure_ascii=False, indent=2)
        logs.append(f'✅ {out_path} ({os.path.getsize(out_path):,} bytes)')
        return out_path, '\n'.join(logs)
    except Exception as e:
        logs.append(f'❌ {traceback.format_exc()}')
        return None, '\n'.join(logs)

# ── Tab 4: Metrics ─────────────────────────────────────────────────────
def run_metrics(ref_text, hyp_text):
    if not ref_text.strip() or not hyp_text.strip():
        return '⚠️ أدخل النصين', ''
    logs = []
    result = {}
    try:
        from modules.evaluation.metrics import compute_cer, compute_wer
        result['CER'] = round(compute_cer(ref_text, hyp_text), 4)
        result['WER'] = round(compute_wer(ref_text, hyp_text), 4)
        result['دقة الحروف'] = f'{(1-result["CER"])*100:.1f}%'
        result['دقة الكلمات'] = f'{(1-result["WER"])*100:.1f}%'
        grade = 'A+' if result['CER']<0.02 else 'A' if result['CER']<0.05 else 'B' if result['CER']<0.10 else 'C' if result['CER']<0.20 else 'F'
        result['التقييم'] = grade
        logs.append('✅ اكتمل')
    except Exception as e:
        from modules.nlp.arabic_nlp_utils import arabic_normalized_similarity
        sim = arabic_normalized_similarity(ref_text, hyp_text)
        result['التشابه (بديل)'] = round(sim, 4)
        logs.append(f'⚠️ metrics غير متاح، استُخدم بديل: {e}')
    return json.dumps(result, ensure_ascii=False, indent=2), '\n'.join(logs)

# ── Tab 5: Health Dashboard ─────────────────────────────────────────────
def run_health_check():
    import importlib
    modules = [
        ('modules.core.structure',          'Core Models'),
        ('modules.vision.ocr_engine',       'OCR Engine'),
        ('modules.vision.image_preprocessor','Image Preprocessor'),
        ('modules.vision.layout_analyzer',  'Layout Analyzer'),
        ('modules.nlp.arabic_rtl',          'Arabic RTL'),
        ('modules.nlp.spell_corrector',     'Spell Corrector'),
        ('modules.nlp.arabic_nlp_utils',    'Arabic NLP Utils 🆕'),
        ('modules.export.exporter',         'Exporter'),
        ('modules.export.layout_preserving','Layout Preserving 🆕'),
        ('modules.evaluation.metrics',      'Metrics'),
        ('modules.security.encryption',     'Encryption'),
        ('modules.ai.pattern_matcher',      'Pattern Matcher'),
    ]
    lines = ['📊 حالة وحدات المشروع\n' + '='*45]
    ok = fail = 0
    for mod, label in modules:
        try:
            importlib.import_module(mod)
            lines.append(f'✅ {label}')
            ok += 1
        except Exception as e:
            lines.append(f'❌ {label}\n   └ {str(e)[:70]}')
            fail += 1
    lines.append('='*45)
    lines.append(f'النتيجة: {ok} ✅  {fail} ❌  المجموع: {ok+fail}')
    pct = ok/(ok+fail)*100 if (ok+fail) else 0
    lines.append(f'الصحة العامة: {pct:.0f}%  {"🟢" if pct>=70 else "🟡" if pct>=50 else "🔴"}')
    return '\n'.join(lines)

print('✅ دوال Gradio جاهزة')

In [ ]:
# ── بناء واجهة Gradio ─────────────────────────────────────────────────
with gr.Blocks(
    title='🧠 OmniFile Diagnostic',
    theme=gr.themes.Soft(primary_hue='blue', neutral_hue='slate'),
    css='.gradio-container{direction:rtl;font-family:Arial,sans-serif}'
) as demo:

    gr.Markdown('# 🧠 OmniFile AI Processor — لوحة الاختبار التشخيصي')
    gr.Markdown('**المطور:** Dr Abdulmalek Tamer Al-husseini | اختبر الوحدات مباشرةً من الجوال أو الحاسوب')

    with gr.Tab('🔍 OCR'):
        with gr.Row():
            with gr.Column(scale=1):
                img_in  = gr.Image(label='📸 ارفع الصورة', type='numpy')
                engine  = gr.Radio(['EasyOCR','Tesseract','OmniFile Engine'], value='EasyOCR', label='المحرك')
                preproc = gr.Checkbox(label='معالجة مسبقة (Otsu)', value=True)
                ocr_btn = gr.Button('🚀 تشغيل OCR', variant='primary')
            with gr.Column(scale=1):
                ocr_out  = gr.Textbox(label='📄 النص المستخرج', lines=8, rtl=True)
                ocr_log  = gr.Textbox(label='📋 السجل', lines=5)
        ocr_btn.click(run_ocr, [img_in, engine, preproc], [ocr_out, ocr_log])

    with gr.Tab('🗣️ NLP'):
        with gr.Row():
            with gr.Column(scale=1):
                nlp_in  = gr.Textbox(label='📝 النص (سطرين للمقارنة)', lines=5, rtl=True,
                                     placeholder='للمقارنة: سطر1\nسطر2')
                nlp_op  = gr.Dropdown(
                    choices=['تطبيع عربي','تقرير تشابه','RTL معالجة','كشف لغة','تدقيق إملائي'],
                    value='تطبيع عربي', label='العملية'
                )
                nlp_btn = gr.Button('🚀 تشغيل', variant='primary')
            with gr.Column(scale=1):
                nlp_out = gr.Textbox(label='النتيجة', lines=8, rtl=True)
                nlp_log = gr.Textbox(label='السجل', lines=4)
        nlp_btn.click(run_nlp, [nlp_in, nlp_op], [nlp_out, nlp_log])

    with gr.Tab('📤 تصدير'):
        with gr.Row():
            with gr.Column(scale=1):
                exp_in  = gr.Textbox(label='📝 النص المراد تصديره', lines=6, rtl=True)
                exp_fmt = gr.Radio(['DOCX','TXT','JSON'], value='TXT', label='الصيغة')
                exp_btn = gr.Button('💾 تصدير', variant='primary')
            with gr.Column(scale=1):
                exp_file = gr.File(label='⬇️ تنزيل الملف')
                exp_log  = gr.Textbox(label='السجل', lines=5)
        exp_btn.click(run_export, [exp_in, exp_fmt], [exp_file, exp_log])

    with gr.Tab('📊 مقاييس الجودة'):
        with gr.Row():
            met_ref = gr.Textbox(label='النص المرجعي', lines=4, rtl=True)
            met_hyp = gr.Textbox(label='نص OCR (للمقارنة)', lines=4, rtl=True)
        met_btn = gr.Button('📐 احسب CER/WER', variant='primary')
        with gr.Row():
            met_out = gr.Textbox(label='النتائج', lines=8, scale=1)
            met_log = gr.Textbox(label='السجل',   lines=4, scale=1)
        met_btn.click(run_metrics, [met_ref, met_hyp], [met_out, met_log])

    with gr.Tab('🩺 صحة المشروع'):
        health_btn = gr.Button('🔄 فحص الآن', variant='primary', size='lg')
        health_out = gr.Textbox(label='تقرير الصحة', lines=20, interactive=False)
        health_btn.click(run_health_check, [], [health_out])

print('✅ واجهة Gradio جاهزة للتشغيل')

In [ ]:
# ── تشغيل الواجهة ───────────────────────────────────────────────────────
# share=True → رابط عام يعمل من الجوال (صالح ساعتين)
demo.launch(
    share=True,
    server_name='0.0.0.0',
    server_port=7860,
    show_error=True,
    quiet=False
)
# ستظهر روابط بهذا الشكل:
# Running on local URL:  http://0.0.0.0:7860
# Running on public URL: https://XXXXXX.gradio.live  ← هذا الرابط للجوال

## 📱 الخطوة 6 — الوصول من الجوال عبر ngrok (بديل)
> إذا انتهت صلاحية رابط Gradio (ساعتين)، استخدم ngrok للوصول الدائم من الجوال.


In [ ]:
# تثبيت pyngrok إذا أردت رابطاً دائماً من الجوال
# 1. سجّل في https://ngrok.com مجاناً واحصل على TOKEN
# 2. ضع التوكن هنا:

NGROK_TOKEN = ''  # ← ضع توكنك هنا

if NGROK_TOKEN:
    !pip install -q pyngrok
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(7860)
    print(f'🌐 رابط الجوال الدائم: {public_url}')
else:
    print('ℹ️ لتفعيل ngrok: أدخل NGROK_TOKEN أعلاه')
    print('   الرابط المؤقت من Gradio (ساعتين) يكفي للاختبار')

## 🖥️ الخطوة 7 — تشغيل محلي على الحاسوب (Manjaro)

In [ ]:
# هذه الخلية للتذكير فقط — نفّذها في ترمينال الحاسوب، ليس هنا
COMMANDS = '''
# ── الخطوات على Manjaro (xorthomson) ────────────────────────────────

# 1. استنساخ أو تحديث
cd ~/Projects && git clone https://github.com/DrAbdulmalek/OmniFile_Processor.git
cd OmniFile_Processor

# 2. بيئة افتراضية
python -m venv .venv && source .venv/bin/activate

# 3. تثبيت خفيف أولاً
pip install gradio easyocr python-docx arabic-reshaper python-bidi langdetect

# 4. تشغيل واجهة Gradio
python -m src.gradio_ui

# 5. أو Streamlit
pip install streamlit && streamlit run app.py

# 6. للوصول من الجوال على نفس الشبكة
python -m src.gradio_ui --server-name 0.0.0.0 --server-port 7860
# ثم افتح من الجوال: http://<IP-الحاسوب>:7860
# لمعرفة الـ IP:
ip addr show | grep 'inet ' | grep -v 127
'''
print(COMMANDS)

## 🐛 الخطوة 8 — خلايا تشخيص سريع

In [ ]:
# 📂 عرض هيكل المشروع الفعلي (مقارنة مع README)
import os
base = '/content/OmniFile_Processor'
for root, dirs, files in os.walk(base):
    dirs[:] = [d for d in dirs if not d.startswith('.') and d not in ['__pycache__','node_modules','.git']]
    level = root.replace(base, '').count(os.sep)
    if level > 3:
        continue
    indent = '  ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')
    if level < 3:
        for f in sorted(files)[:8]:
            print(f'{indent}  📄 {f}')
        if len(files) > 8:
            print(f'{indent}  ... و{len(files)-8} ملف آخر')

In [ ]:
# 🔍 فحص أي ملف أو وحدة بشكل مباشر
FILE_TO_INSPECT = 'modules/nlp/arabic_nlp_utils.py'  # ← غيّر هنا

path = os.path.join('/content/OmniFile_Processor', FILE_TO_INSPECT)
if os.path.exists(path):
    print(f'📄 {FILE_TO_INSPECT} ({os.path.getsize(path):,} bytes):')
    print('─'*50)
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read()
    print(content[:2000])
    if len(content) > 2000:
        print(f'... [{len(content)-2000} حرف إضافي]')
else:
    print(f'❌ الملف غير موجود: {path}')